In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from pybaseball import statcast, playerid_lookup

In [ ]:
START_DATE = '2023-03-30'
END_DATE = '2026-09-27'
TEST_START_DATE = '2026-03-25'
CACHE_DIR = Path('data')
CACHE_PATH = CACHE_DIR / f'statcast_{START_DATE}_{END_DATE}.parquet'

if CACHE_PATH.exists():
    raw = pd.read_parquet(CACHE_PATH)
    print(f'Loaded {len(raw):,} pitches from {CACHE_PATH}')
else:
    CACHE_DIR.mkdir(parents = True, exist_ok = True)
    raw = statcast(START_DATE, END_DATE)
    raw.to_parquet(CACHE_PATH, index = False)
    print(f'Downloaded and cached {len(raw):,} pitches to {CACHE_PATH}')

This is a large query, it may take a moment to complete


/Users/evanfroehlich/Documents/Baseball projects/mlb_expected_stats_model/.venv/lib/python3.13/site-packages/pybaseball/statcast.py:50: UserWarning: 
That's a nice request you got there. It'd be a shame if something were to happen to it.
We strongly recommend that you enable caching before running this. It's as simple as `pybaseball.cache.enable()`.
Since the Statcast requests can take a *really* long time to run, if something were to happen, like: a disconnect;
gremlins; computer repair by associates of Rudy Giuliani; electromagnetic interference from metal trash cans; etc.;
you could lose a lot of progress. Enabling caching will allow you to immediately recover all the successful
subqueries if that happens.
  warnings.warn(_OVERSIZE_WARNING)


Skipping offseason dates
Skipping offseason dates
Skipping offseason dates


100%|██████████| 920/920 [01:01<00:00, 14.88it/s]


Downloaded and cached 3018395 pitches to data/statcast_2023-03-30_2026-09-27.parquet


In [7]:
bbe = {'field_out', 'single', 'double', 'home_run', 'force_out', 'grounded_into_double_play',
       'field_error', 'sac_fly', 'triple', 'fielders_choice', 'double_play', 'fielders_choice_out',
       'sac_fly_double_play', 'triple_play'}

In [16]:
contact = raw.loc[
    (raw.events.isin(bbe)) & (raw.game_type == 'R'),
    ['game_date', 'batter', 'events', 'hc_x', 'hc_y', 'hit_distance_sc', 'launch_speed', 
     'launch_angle', 'stand', 'p_throws', 'pitch_type']]

contact['spray_angle'] = np.degrees(
    np.arctan2(contact["hc_x"] - 125.42, 198.27 - contact["hc_y"])
)

contact['is_hit'] = contact["events"].isin(
    {"single", "double", "triple", "home_run"}
).astype(int)

contact.head()

,game_date,batter,events,hc_x,hc_y,hit_distance_sc,launch_speed,launch_angle,stand,p_throws,pitch_type,spray_angle,is_hit
3058,2026-09-27,811965,field_out,106.88,84.45,288,90.1,45,R,R,SL,-9.251587,0
3291,2026-09-27,695720,field_out,167.48,128.36,205,68.5,46,R,R,SL,31.032406,0
2431,2026-09-27,664774,field_out,124.99,187.49,2,64.4,-53,L,L,ST,-2.284242,0
2778,2026-09-27,665161,field_out,149.99,64.47,339,88.4,30,R,L,FF,10.405429,0
3525,2026-09-27,678489,field_out,118.37,209.96,40,79.6,82,R,L,FF,-148.90668,0


In [17]:
model_data = contact.dropna(
    subset=["launch_speed", "launch_angle", "spray_angle", 
            "stand", "p_throws", "pitch_type", "is_hit"]
).copy()

print(f"Contact events: {len(contact):,}")
print(f"Complete rows: {len(model_data):,}")
print(f"Observed batting average: {model_data['is_hit'].mean():.3f}")
print("Missing values by field:")
print(contact.isna().sum().sort_values(ascending=False))

Contact events: 495,813
Complete rows: 493,896
Observed batting average: 0.324
Missing values by field:
launch_speed       1659
hit_distance_sc    1644
launch_angle       1358
hc_x                211
hc_y                211
spray_angle         211
pitch_type          187
game_date             0
batter                0
events                0
stand                 0
p_throws              0
is_hit                0
dtype: int64


In [18]:
model_data.game_date = pd.to_datetime(model_data.game_date)
test_start_date = pd.Timestamp(TEST_START_DATE)
train_data = model_data[model_data['game_date'] < test_start_date].copy()
test_data = model_data[model_data['game_date'] >= test_start_date].copy()

In [19]:
numeric_features = ['hit_distance_sc', 'launch_speed', 'launch_angle', 'spray_angle']
categorical_features = ['stand', 'p_throws', 'pitch_type']
features = numeric_features + categorical_features

X_train = train_data[features]
X_test = train_data['is_hit']
y_train = test_data[features]
y_test = test_data['is_hit']

print(f"Training rows: {len(train_data):,}")
print(f"Test rows: {len(test_data):,}")
print(f"Training dates: {train_data['game_date'].min().date()} to {train_data['game_date'].max().date()}")
print(f"Test dates: {test_data['game_date'].min().date()} to {test_data['game_date'].max().date()}")

Training rows: 370,575
Test rows: 123,321
Training dates: 2023-03-30 to 2025-09-28
Test dates: 2026-03-25 to 2026-09-27
